# Task: Convolution and Deconvolution

**Student Name:**  Sandra Senn

**Country:**  Morocco

**Semester term:** FS26  

**Repository:** https://github.com/Sandra-Senn/gbsv_mc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile as wav
import IPython.display as ipd

In [ ]:
def load_audio(path='../mc1/data/dromedar.wav'):
    """Load, mono-convert and normalize a WAV file.
    Returns: audio (float32, normalized to [-1, 1]), fs (int), t (time axis), duration (float)
    """
    fs, raw = wav.read(path)
    if raw.ndim == 2:
        raw = raw.mean(axis=1).astype(raw.dtype)
    audio = raw.astype(np.float32) / np.iinfo(raw.dtype).max
    t = np.arange(len(audio)) / fs
    duration = len(audio) / fs
    return audio, fs, t, duration


def gaussian_kernel(n, sigma):
    x = np.arange(n) - n // 2
    g = np.exp(-x**2 / (2 * sigma**2))
    return g / g.sum()


def manual_convolve(x, kernel):
    """1D convolution with zero-padding boundary handling (same-length output).
    Implements: y[n] = sum_k h[k] * x[n - k] via sliding window.
    """
    n, m   = len(x), len(kernel)
    pad    = m // 2
    padded = np.pad(x, pad, mode='constant')
    kernel_flipped = kernel[::-1]    # convolution flips the kernel
    out    = np.zeros(n)
    for i in range(n):
        out[i] = np.dot(padded[i:i+m], kernel_flipped)
    return out


def wiener_deconv(filtered, kernel, lam, N):
    """Wiener deconvolution: H_inv = H* / (|H|^2 + lambda).
    lambda controls regularisation: small = sharper but noisier reconstruction.
    """
    H     = np.fft.rfft(kernel, n=N)
    Y     = np.fft.rfft(filtered)
    H_inv = np.conj(H) / (np.abs(H)**2 + lam)
    return np.fft.irfft(Y * H_inv, n=N)


def spec_db(x, ref): 
    spec_db = 20*np.log10(np.abs(np.fft.rfft(x))/ref + 1e-10)
    return spec_db


def plot_kernel_analysis(ax_res, ax_spec, ax_ver,
                         audio, t, freqs, seg,
                         filtered, rec, man_conv, lib_conv,
                         sc_orig, sc_f, sc_r,
                         zoom_s, zoom_e,
                         color_f, color_r, label):
    """Reusable plot function for one kernel configuration.
    All required arrays are passed explicitly to avoid reliance on global scope.
    """
    ref_max = np.abs(np.fft.rfft(audio)).max()
    mask    = (t >= zoom_s) & (t <= zoom_e)
    t_seg   = np.arange(len(seg)) / (len(t) / t[-1]) * 1000  # ms

    def spec_db(x): return 20 * np.log10(np.abs(np.fft.rfft(x)) / ref_max + 1e-10)

    # Residual — zoomed to cry peak
    # Note: time-domain smoothing is sub-millisecond and invisible at full scale;
    # residual makes the filtering and reconstruction error explicitly visible
    diff_f = audio - filtered
    diff_r = audio - rec
    ax_res.plot(t[mask], diff_f[mask], color=color_f, lw=0.8,
                label=f'Original − Filtered ({label})')
    ax_res.plot(t[mask], diff_r[mask], color=color_r, lw=0.8, linestyle='--',
                label=f'Original − Reconstructed ({label})')
    ax_res.axhline(y=0, color='black', lw=0.8, linestyle=':')
    ax_res.set_ylabel('Residual [norm.]')
    ax_res.set_xlabel('Time [s]')
    ax_res.set_title(f'Residual zoomed to cry peak ({zoom_s}–{zoom_e} s) — {label}')
    ax_res.legend(fontsize=8)
    ax_res.grid(True, alpha=0.15)

    # Spectrum — zoomed to 0–6000 Hz
    # Note: zoomed to 0–6000 Hz to clearly show harmonic attenuation and recovery
    ax_spec.plot(freqs, spec_db(audio),    color='steelblue', lw=0.7, label='Original')
    ax_spec.plot(freqs, spec_db(filtered), color=color_f,     lw=0.9,
                 label=f'Filtered (SC_diff={sc_f:.0f} Hz)')
    ax_spec.plot(freqs, spec_db(rec),      color=color_r,     lw=0.9, linestyle='--',
                 label=f'Reconstructed (SC_diff={sc_r:.0f} Hz)')
    for sc_val, col in [(sc_orig, 'steelblue'),
                        (sc_orig + sc_f, color_f),
                        (sc_orig + sc_r, color_r)]:
        ax_spec.axvline(x=sc_val, color=col, linestyle=':', lw=1, alpha=0.6)
    ax_spec.set_xlim([0, 6000])
    ax_spec.set_ylim([-80, 5])
    ax_spec.set_xlabel('Frequency [Hz]')
    ax_spec.set_ylabel('Magnitude [dB]')
    ax_spec.set_title(f'Power spectrum zoomed to 0–6 000 Hz — {label} (dotted = SC)')
    ax_spec.legend(fontsize=8)
    ax_spec.grid(True, alpha=0.3)

    # Manual vs library verification — cry segment
    ax_ver.plot(t_seg, seg,      color='steelblue', lw=0.6, alpha=0.7, label='Original (cry onset)')
    ax_ver.plot(t_seg, man_conv, color=color_f,     lw=1.5, label=f'Manual conv ({label})')
    ax_ver.plot(t_seg, lib_conv, color=color_r,     lw=0.9, linestyle='--',
                label='Library conv (verify – overlaps manual)')
    ax_ver.set_xlabel('Time [ms]')
    ax_ver.set_ylabel('Amplitude [norm.]')
    ax_ver.set_title(f'Manual vs library verification — {label}: outputs overlap exactly')
    ax_ver.legend(fontsize=8)
    ax_ver.grid(True, alpha=0.15)

def spectral_centroid(x, freqs):
    s = np.abs(np.fft.rfft(x))
    sc = np.sum(freqs * s) / np.sum(s)
    return sc

## Morocco - Dromedar Market

<p style="display: flex; align-items: center;">
  <img src="data/img/family.jpeg"  description="camel_family" style="width:350px; margin-right:20px;">
  <span style="flex: 1;">
  The same 2.4 seconds. The same piercing cry cutting through the market chaos. 
  But this week, I'm asking a completely different question — not <em>how fast to sample it</em>, not <em>where to find it in the noise</em>, but: <strong>can we clean it up?</strong>
  The Guelmim market is loud. Wind, haggling voices, bleating animals — all of it is baked into the recording. What if a zoo veterinarian needs to analyse the harmonic structure of that cry, but the noise gets in the way? That's where convolution comes in: a controlled smoothing operation that attenuates the noise — and deconvolution, its imperfect but fascinating inverse, that tries to undo what was done.
  Welcome to Week 3.
  </span>
</p>

## Day 11 – Data & Domain

### Use Case
*Focus: domain and application context*

In the context of acoustic welfare monitoring at Swiss Zoos, the dromedary vocalization recorded at the Guelmim camel market is contaminated by broadband market noise — wind, crowd chatter, and human voices — that obscures the cry's harmonic structure and complicates spectral feature extraction.

These recordings are processed by veterinarians and zoo staff to perform non-invasive stress-state classification; for this, a smoothed, noise-reduced version of the signal is more interpretable than the raw field recording.

This use case is particularly relevant for Switzerland because zoo monitoring systems deployed in the field must be robust to environmental noise, making controlled smoothing via convolution — and the subsequent recovery of attenuated features via deconvolution — a practically motivated signal processing challenge.

### Problem Statement
*Focus: technical vulnerability*

This project addresses the problem of applying one-dimensional convolution with smoothing kernels to the dromedary cry signal in order to attenuate broadband market noise, within the context of acoustic welfare monitoring at Zoos.

If the kernel is too narrow, high-frequency noise persists and harmonic features remain obscured. If it is too wide, dominant cry harmonics are smoothed away, altering the timbral structure and reducing the informativeness of the signal for welfare assessment.

Preserving the spectral shape of the cry's harmonic content is essential for reliable stress-state classification — making deconvolution relevant as a means of recovering attenuated features after filtering, provided the kernel is sufficiently well-characterised.


### Experimental Objective
*Focus: investigation goal at the conceptual level.*

This investigation aims to **(1)** implement one-dimensional convolution manually and apply it with Gaussian and moving-average kernels of varying sizes to the dromedary cry signal, assessing the trade-off between noise attenuation and harmonic preservation. And **(2)** apply Wiener deconvolution to the filtered signal to reconstruct an approximation of the original, evaluating how effectively the filtering effect can be reversed and under which conditions reconstruction fidelity is practically useful for the welfare monitoring application.

### Data Definition, Source, and Visualization
*Focus: data characteristics, data source, and visual inspection.*

The selected signal is the same baby dromedary vocalization (field recording, Guelmim camel market, Morocco, February 2026) used throughout this Mini Challenge — a mono WAV file at **44 100 Hz**, **2.415 s**, normalized to amplitude ∈ [−1, +1]. The signal contains a dominant cry burst between **1.39–2.08 s** (identified via the energy-ratio method in Week 2) with fundamental frequency **f₀ ≈ 428 Hz** and spectral centroid at **2 182 Hz**, embedded in broadband market noise. The high-frequency noise components above ~6 000 Hz and the low-frequency market rumble below ~200 Hz are the target for smoothing, while the harmonic series (428–4 708 Hz) must be preserved for welfare assessment. The signal's known spectral structure from the Sampling Theorem analysis (Week 1) provides a well-characterized baseline for evaluating filtering and reconstruction quality.

In [ ]:
# Load audio extracted from the Guelmim market video 
audio, fs, t, duration = load_audio()

# get Baseline Gaussian kernel
k_baseline = gaussian_kernel(21, sigma=3)
filtered   = np.convolve(audio, k_baseline, mode='same')


#  Spectral analysis 
freqs    = np.fft.rfftfreq(len(audio), 1/fs)
spec_orig = np.abs(np.fft.rfft(audio))
spec_filt = np.abs(np.fft.rfft(filtered))
sc_orig = spectral_centroid(audio,    freqs)
sc_filt = spectral_centroid(filtered, freqs)

In [ ]:
#  Visualization 
fig, axes = plt.subplots(3, 1, figsize=(13, 10))
fig.suptitle('Baby Dromedary Cry – Signal Overview for Convolution Analysis\n'
             'Guelmim Camel Market, Morocco (field recording, Feb 2026)',
             fontsize=13, fontweight='bold')

# Waveform
axes[0].plot(t, audio, color='steelblue', linewidth=0.4, label='Original')
axes[0].plot(t, filtered, color='crimson', linewidth=0.8, alpha=0.85,
             label='Filtered (Gauss-21, σ=3) – baseline')
axes[0].axvspan(1.39, 2.08, color='orange', alpha=0.15, label='Cry region')
axes[0].set_ylabel('Amplitude [norm.]')
axes[0].set_title('Waveform: original vs. baseline-filtered signal')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Power spectrum
axes[1].plot(freqs, 20*np.log10(spec_orig/spec_orig.max() + 1e-10),
             color='steelblue', linewidth=0.7, label='Original')
axes[1].plot(freqs, 20*np.log10(spec_filt/spec_orig.max() + 1e-10),
             color='crimson',   linewidth=0.8, label='Filtered (Gauss-21, σ=3)')
axes[1].axvline(x=sc_orig, color='steelblue', linestyle='--', linewidth=1,
                label=f'SC original = {sc_orig:.0f} Hz')
axes[1].axvline(x=sc_filt, color='crimson',   linestyle='--', linewidth=1,
                label=f'SC filtered = {sc_filt:.0f} Hz')
axes[1].set_xlim([0, 8000])
axes[1].set_ylim([-80, 5])
axes[1].set_xlabel('Frequency [Hz]')
axes[1].set_ylabel('Magnitude [dB]')
axes[1].set_title('Power spectrum: effect of baseline smoothing on harmonic structure')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# Kernel shape
axes[2].stem(np.arange(len(k_baseline)) - len(k_baseline)//2, k_baseline,
             linefmt='crimson', markerfmt='ro', basefmt='k-')
axes[2].set_xlabel('Sample offset')
axes[2].set_ylabel('Weight')
axes[2].set_title('Baseline kernel: Gaussian (n=21, σ=3)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print(f'Signal length        : {len(audio)} samples ({len(audio)/fs:.3f} s) @ {fs} Hz')
print(f'Spectral centroid    : {sc_orig:.1f} Hz (original)')
print(f'Spectral centroid    : {sc_filt:.1f} Hz (filtered, Gauss-21)')
print(f'SC shift             : {sc_filt - sc_orig:.1f} Hz')

print('\nOriginal recording:')
ipd.display(ipd.Audio(audio, rate=fs))
print('Filtered (Gauss-21, σ=3):')
ipd.display(ipd.Audio(filtered, rate=fs))

**Observations**:

The visualization illustrates the original dromedary field recording alongside its baseline-filtered version (Gaussian kernel, n=21, σ=3), with notable amplitude reduction and waveform smoothing observed throughout the full 2.415 s signal. The power spectrum reveals that baseline smoothing attenuates high-frequency content above ~3 000 Hz, shifting the spectral centroid from **2 182 Hz** to **1 222 Hz** (−960 Hz), while the dominant harmonic at f₀ ≈ 428 Hz remains visible. This region was selected because it clearly highlights the trade-off between noise attenuation and harmonic preservation directly relevant to the welfare monitoring use case and the convolution investigation objective.


## Day 12 – Methodological Design

### Theoretical Foundation and Method Choice
*Focus: principled justification aligned with the use case*

This investigation applies **one-dimensional linear convolution** as a smoothing operation and **Wiener deconvolution** as its inverse within the context of acoustic welfare monitoring of dromedaries in Zoos. Convolution with a low-pass kernel acts as a linear time-invariant (LTI) filter that attenuates high-frequency noise while preserving lower-frequency harmonic content — its validity rests on the assumptions of linearity and shift-invariance of the recording system, which are reasonable for a short, quasi-stationary cry segment. Wiener deconvolution is selected as the inverse method because it minimises the mean squared reconstruction error in the frequency domain via

$$
H^{-1}(\omega) = \frac{H^*(\omega)}{|H(\omega)|^2 + \lambda}
$$

where $\lambda$ is a regularisation parameter that prevents noise amplification at frequencies where the kernel has near-zero gain — making it preferable over naive inverse filtering which is unstable for smooth kernels. If the stationarity assumption is violated (e.g., rapid amplitude changes during the cry onset), the LTI model becomes an approximation, and convolution may introduce transient distortions at signal boundaries.

### Parameter Definition and Mathematical Specification
*Focus: explicit parameter selection, derivation, and unit consistency*

**Convolution kernels:**

| Kernel | n | σ / description | RMSE | SC_diff [Hz] | Justification |
|:---|:---|:---|:---|:---|:---|
| Moving average | 5 | uniform | 0.00414 | −188 | Minimal smoothing reference |
| Moving average | 21 | uniform | 0.01981 | −1281 | Moderate; known zero-phase issue |
| **Gaussian (baseline)** | **21** | **σ=3** | **0.01125** | **−960** | **Best noise/harmonic trade-off** |
| Gaussian | 51 | σ=8 | 0.01879 | −1673 | Stronger smoothing |
| Gaussian | 101 | σ=15 | 0.02277 | −1840 | Maximum smoothing tested |

The Gaussian kernel is selected over the moving average as the baseline because it has a smooth frequency response without the spectral sidelobes of the rectangular window, preserving more harmonic energy at the cost of less sharp high-frequency cutoff. The baseline kernel size n=21 (≈ 0.48 ms at 44 100 Hz) is small relative to the fundamental period T₀ = 1/428 Hz = 2.34 ms, ensuring that the kernel does not blur across full pitch cycles.

**Wiener deconvolution parameter:**

| Parameter | Symbol | Baseline value | Range tested | Physical meaning |
|:---|:---|:---|:---|:---|
| Regularisation | λ | 1×10⁻³ | 1e-6 – 1e-1 | Controls noise amplification vs. reconstruction fidelity |

The baseline λ = 10⁻³ is chosen because it provides a reasonable trade-off: 
- reconstruction RMSE = 0.03309 
-  SC_diff = −200.8 Hz — recovering ~76 % of the spectral centroid shift introduced by filtering (original shift −960 Hz, recovered to −201 Hz). 

For welfare assessment, a SC_diff < 300 Hz post-reconstruction is considered practically acceptable as it preserves the dominant harmonic region.

### Experimental Design for Next Days
*Focus: structured parameter variation and theoretical prediction*

The **baseline configuration** is defined as: 

Gaussian kernel (n=21, σ=3) for convolution, Wiener deconvolution with λ=10⁻³, applied to the full 2.415 s signal.

| Day | Parameter | Values | Prediction |
|:---|:---|:---|:---|
| 13 | Kernel type | MA-21 vs Gauss-21 | Gauss produces less spectral sidelobe distortion |
| 14 | Kernel size n (Gaussian) | 5, 11, 21, 51, 101 | RMSE_filt increases; SC_diff more negative |
| 14 | Wiener λ | 1e-6, 1e-4, 1e-3, 1e-2, 1e-1 | Small λ -> better spectral recovery but noise amplification |
| 13 | Manual vs library conv | — | Max error < 1e-15 (floating point precision only) |

It is theoretically expected that increasing kernel size will progressively attenuate higher harmonics (wider low-pass cutoff), increasing both RMSE and the magnitude of SC_diff. For Wiener deconvolution, decreasing λ should recover more high-frequency content (SC_diff closer to 0) at the risk of amplifying noise, while increasing λ produces a more stable but spectrally incomplete reconstruction.


### Methodological Limitations and Risk Factors
*Focus: assumptions, stability, and potential misinterpretation*

The LTI convolution model assumes a stationary signal, which is violated by the dromedary cry's rapid amplitude envelope (onset at t=1.39 s) — causing boundary transients in the filtered signal that are not representative of steady-state filtering behavior and may inflate RMSE near the cry onset. Wiener deconvolution assumes that the filtering kernel is exactly known, which holds in this controlled experiment but would not hold in real zoo deployments where the acoustic transfer function of the environment is unknown — limiting the practical applicability of the reconstruction. The spectral centroid is used as a proxy for harmonic preservation, but it captures only the average frequency weighting; it may underestimate distortion at specific harmonics critical for stress-state classification.

## Day 13 – Implementation

*Focus: structured, traceable execution*


In [ ]:
audio, fs, t, duration = load_audio()
N = len(audio)

#  Kernels 
k_gauss21 = gaussian_kernel(21, sigma=3)
k_gauss51 = gaussian_kernel(51, sigma=8)   # larger kernel for visible smoothing effect
k_ma21    = np.ones(21) / 21

In [ ]:
#  Manual convolution on cry segment for verification 
# Segment taken from cry region (not silence) so differences are visible
cry_start = int(1.39 * fs)
cry_end   = int(1.49 * fs)   # 100 ms of active cry
seg       = audio[cry_start:cry_end]


man_g21  = manual_convolve(seg, k_gauss21)
man_g51  = manual_convolve(seg, k_gauss51)
lib_g21  = np.convolve(seg, k_gauss21, mode='same')
lib_g51  = np.convolve(seg, k_gauss51, mode='same')



print('Manual vs library verification (max absolute error):')
print(f'  Gaussian-21: {np.max(np.abs(man_g21 - lib_g21)):.2e}')
print(f'  Gaussian-51: {np.max(np.abs(man_g51 - lib_g51)):.2e}')
print('  → Errors are at floating-point precision (< 1e-15), confirming correctness.')

In [ ]:
#  Apply to full signal 
filtered_g21  = np.convolve(audio, k_gauss21, mode='same')
filtered_g51  = np.convolve(audio, k_gauss51, mode='same')
filtered_ma21 = np.convolve(audio, k_ma21,    mode='same')


# Both kernels reconstructed — key comparison: Gauss-21 is partially invertible,
# Gauss-51 is not, because stronger smoothing destroys more spectral information
rec_g21 = wiener_deconv(filtered_g21, k_gauss21, lam=1e-3, N=N)
rec_g51 = wiener_deconv(filtered_g51, k_gauss51, lam=1e-3, N=N)

#  Metrics 
freqs   = np.fft.rfftfreq(N, 1/fs)
sc      = lambda x: np.sum(freqs * np.abs(np.fft.rfft(x))) / np.sum(np.abs(np.fft.rfft(x)))
sc_orig    = sc(audio)
sc_f21     = sc(filtered_g21)
sc_f51     = sc(filtered_g51)
sc_rec_g21 = sc(rec_g21)
sc_rec_g51 = sc(rec_g51)

print('Baseline Gauss-21 (σ=3, λ=1e-3):')
print(f'  RMSE filtered:      {np.sqrt(np.mean((audio - filtered_g21)**2)):.5f}')
print(f'  RMSE reconstructed: {np.sqrt(np.mean((audio - rec_g21)**2)):.5f}')
print(f'  SC shift filtered:      {sc_f21   - sc_orig:.1f} Hz')
print(f'  SC shift reconstructed: {sc_rec_g21 - sc_orig:.1f} Hz')
print('\nGauss-51 (σ=8, λ=1e-3):')
print(f'  RMSE filtered:      {np.sqrt(np.mean((audio - filtered_g51)**2)):.5f}')
print(f'  RMSE reconstructed: {np.sqrt(np.mean((audio - rec_g51)**2)):.5f}')
print(f'  SC shift filtered:      {sc_f51   - sc_orig:.1f} Hz')
print(f'  SC shift reconstructed: {sc_rec_g51 - sc_orig:.1f} Hz')

In [ ]:
#  Visualization 
zoom_s, zoom_e = 1.50, 1.65
mask    = (t >= zoom_s) & (t <= zoom_e)
t_seg   = np.arange(len(seg)) / fs * 1000



#  Figure 1: Gauss-21 (baseline) 
fig1, ax1 = plt.subplots(3, 1, figsize=(13, 11))
fig1.suptitle('Day 13 – Figure 1: Gauss-21 (n=21, σ=3) — Baseline Kernel\n'
              'Mild smoothing, partially invertible via Wiener deconvolution',
              fontsize=12, fontweight='bold')
plot_kernel_analysis(ax1[0], ax1[1], ax1[2],
                     audio, t, freqs, seg,
                     filtered_g21, rec_g21, man_g21, lib_g21,
                     sc_orig, sc_f21 - sc_orig, sc_rec_g21 - sc_orig,
                     zoom_s=1.50, zoom_e=1.65,
                     color_f='crimson', color_r='seagreen',
                     label='Gauss-21')
plt.tight_layout()
plt.show()

print('\nFiltered signal (Gauss-21):')
ipd.display(ipd.Audio(filtered_g21, rate=fs))
print('Reconstructed from Gauss-21 (Wiener):')
ipd.display(ipd.Audio(rec_g21, rate=fs))


In [ ]:
#  Figure 2: Gauss-51 (stronger smoothing) 
fig2, ax2 = plt.subplots(3, 1, figsize=(13, 11))
fig2.suptitle('Day 13 – Figure 2: Gauss-51 (n=51, σ=8) — Stronger Kernel\n'
              'Heavy smoothing, reconstruction fails — spectral information irreversibly lost',
              fontsize=12, fontweight='bold')
plot_kernel_analysis(ax2[0], ax2[1], ax2[2],
                     audio, t, freqs, seg,
                     filtered_g51, rec_g51, man_g51, lib_g51,
                     sc_orig, sc_f51 - sc_orig, sc_rec_g51 - sc_orig,
                     zoom_s=1.50, zoom_e=1.65,
                     color_f='darkorange', color_r='purple',
                     label='Gauss-51')
plt.tight_layout()
plt.show()


print('\nFiltered signal (Gauss-51):')
ipd.display(ipd.Audio(filtered_g51, rate=fs))
print('Reconstructed from Gauss-51 (Wiener):')
ipd.display(ipd.Audio(rec_g51, rate=fs))


## Day  14 – Evaluation

*Focus: systematic, traceable evaluation of the predefined experiment design and its key parameters.*

### Evaluation Approach Definition

**Metric 1 – RMSE [normalized amplitude]:**

RMSE quantifies point-wise deviation between the original signal and the filtered or reconstructed version — it is chosen because the welfare monitoring use case requires accurate amplitude preservation of the cry waveform, and RMSE directly reflects how much the smoothing operation distorts the temporal structure. A known 44 100 Hz reference is available as ground truth, making point-wise comparison valid.

**Metric 2 – Spectral Centroid Shift SC_diff [Hz]:**

SC_diff = SC_output − SC_original measures the timbral shift introduced by filtering or recovered by deconvolution — it is chosen because harmonic preservation is critical for stress-state classification, and the spectral centroid captures the aggregate effect of low-pass filtering on the harmonic energy distribution. Both metrics are reported for filtered and reconstructed signals to enable a three-way comparison: original → filtered → reconstructed.


### Evaluation Comparison Execution
#### Parameter Evaluation – Kernel Size (Metric-Based)

The influence of **Gaussian kernel size n ∈ {5, 11, 21, 51, 101}** (with σ proportional to n) was evaluated using the previously defined metrics **RMSE** and **SC_diff [Hz]**.
These parameters are essential for the acoustic welfare monitoring use case because they directly affect the trade-off between noise attenuation and harmonic preservation, which is critical for reliable stress-state classification.
Changes in RMSE and SC_diff across kernel sizes quantify how sensitive filtering and reconstruction quality are to the choice of smoothing strength.
Relative performance change was computed with respect to the baseline configuration **Gauss-21, σ=3, λ=10⁻³**.

#### Parameter Evaluation – Wiener Regularisation λ (Visualization-Based)

The influence of **Wiener regularisation λ ∈ {10⁻⁶, 10⁻⁴, 10⁻³, 10⁻², 10⁻¹}** (fixed Gauss-21 kernel) was evaluated using the previously defined visualization-based measure **SC_diff [Hz]** alongside **RMSE**.
This parameter is essential for the defined use case because it influences the spectral recovery of the reconstructed signal, which determines how much of the cry's harmonic structure is restored after filtering in the zoo monitoring context.
Differences in SC_diff and RMSE across λ values quantify the impact of regularisation strength on the stability–fidelity trade-off of the Wiener deconvolution.
Relative performance change was computed with respect to the baseline configuration **λ=10⁻³, Gauss-21, σ=3**.

In [ ]:
#  Experiment 1: Kernel size variation 
kernel_configs = [(5,1), (11,2), (21,3), (51,8), (101,15)]
lam_baseline   = 1e-3
rows_kernel    = []

for n, sigma in kernel_configs:
    k     = gaussian_kernel(n, sigma)
    filt  = np.convolve(audio, k, mode='same')
    rec   = wiener_deconv(filt, k, lam=lam_baseline, N=N)
    rmse_f = np.sqrt(np.mean((audio - filt)**2))
    rmse_r = np.sqrt(np.mean((audio - rec)**2))
    sc_f   = spectral_centroid(filt, freqs)  - sc_orig
    sc_r   = spectral_centroid(rec, freqs)   - sc_orig
    rows_kernel.append((f'Gauss-{n},σ={sigma}', n, rmse_f, rmse_r, sc_f, sc_r))


# Experiment 2: Wiener λ variation 
k_base      = gaussian_kernel(21, sigma=3)
filt_base   = np.convolve(audio, k_base, mode='same')
lambda_vals = [1e-6, 1e-4, 1e-3, 1e-2, 1e-1]
rows_lambda = []

for lam in lambda_vals:
    rec    = wiener_deconv(filt_base, k_base, lam=lam, N=N)
    rmse_r = np.sqrt(np.mean((audio - rec)**2))
    sc_r   = spectral_centroid(rec, freqs) - sc_orig
    rows_lambda.append((lam, rmse_r, sc_r))

In [ ]:
#  Print tables 
print('Experiment 1 – Kernel Size Variation (λ=1e-3):')
print(f"{'Kernel':>16} {'RMSE_filt':>11} {'RMSE_rec':>10} {'SC_diff_filt[Hz]':>17} {'SC_diff_rec[Hz]':>16}")
print('-' * 75)
base_rf, base_rr = rows_kernel[2][2], rows_kernel[2][3]
for name, n, rf, rr, sf, sr in rows_kernel:
    print(f"{name:>16} {rf:>11.5f} {rr:>10.5f} {sf:>17.1f} {sr:>16.1f}")

print('\nExperiment 2 – Wiener λ Variation (Gauss-21, σ=3):')
print(f"{'λ':>10} {'RMSE_rec':>10} {'SC_diff_rec [Hz]':>18} {'Δ RMSE vs baseline':>20}")
print('-' * 62)
base_lam_r = rows_lambda[2][1]
for lam, rr, sr in rows_lambda:
    print(f"{lam:>10.0e} {rr:>10.5f} {sr:>18.1f} {rr-base_lam_r:>+20.5f}")


In [ ]:
#  Visualization 
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Day 14 – Quantitative Evaluation: Kernel Size and Wiener λ Variation',
             fontsize=12, fontweight='bold')

kernel_labels = [f'Gauss-{n}\nσ={s}' for n,s in [(5,1),(11,2),(21,3),(51,8),(101,15)]]
colors_k = ['steelblue','seagreen','crimson','darkorange','purple']

# RMSE vs kernel size
rmse_f_vals = [r[2] for r in rows_kernel]
rmse_r_vals = [r[3] for r in rows_kernel]
x = np.arange(len(kernel_labels))
axes[0,0].bar(x - 0.2, rmse_f_vals, 0.35, color='crimson',  label='RMSE filtered',      edgecolor='black', lw=0.5)
axes[0,0].bar(x + 0.2, rmse_r_vals, 0.35, color='seagreen', label='RMSE reconstructed', edgecolor='black', lw=0.5)
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(kernel_labels, fontsize=8)
axes[0,0].set_ylabel('RMSE')
axes[0,0].set_title('RMSE vs kernel size')
axes[0,0].legend(fontsize=8)
axes[0,0].grid(axis='y', alpha=0.3)

# SC_diff vs kernel size
sc_f_vals = [r[4] for r in rows_kernel]
sc_r_vals = [r[5] for r in rows_kernel]
axes[0,1].bar(x - 0.2, sc_f_vals, 0.35, color='crimson',  label='SC_diff filtered',      edgecolor='black', lw=0.5)
axes[0,1].bar(x + 0.2, sc_r_vals, 0.35, color='seagreen', label='SC_diff reconstructed', edgecolor='black', lw=0.5)
axes[0,1].axhline(y=0, color='black', linewidth=0.8)
axes[0,1].set_xticks(x)
axes[0,1].set_xticklabels(kernel_labels, fontsize=8)
axes[0,1].set_ylabel('SC_diff [Hz]')
axes[0,1].set_title('Spectral centroid shift vs kernel size')
axes[0,1].legend(fontsize=8)
axes[0,1].grid(axis='y', alpha=0.3)

# RMSE vs lambda
lam_labels = [f'λ={l:.0e}' for l,_,_ in rows_lambda]
rmse_lam   = [r[1] for r in rows_lambda]
sc_lam     = [r[2] for r in rows_lambda]
axes[1,0].plot(range(len(lam_labels)), rmse_lam, 'o-', color='seagreen', linewidth=2, markersize=7)
axes[1,0].axhline(y=rows_lambda[2][1], color='gray', linestyle='--', linewidth=1, label='Baseline λ=1e-3')
axes[1,0].set_xticks(range(len(lam_labels)))
axes[1,0].set_xticklabels(lam_labels, fontsize=8)
axes[1,0].set_ylabel('RMSE reconstructed')
axes[1,0].set_title('Reconstruction RMSE vs Wiener λ')
axes[1,0].legend(fontsize=8)
axes[1,0].grid(True, alpha=0.3)

# SC_diff vs lambda
axes[1,1].plot(range(len(lam_labels)), sc_lam, 's-', color='darkorange', linewidth=2, markersize=7)
axes[1,1].axhline(y=0, color='black', linewidth=0.8, label='No shift (perfect reconstruction)')
axes[1,1].set_xticks(range(len(lam_labels)))
axes[1,1].set_xticklabels(lam_labels, fontsize=8)
axes[1,1].set_ylabel('SC_diff [Hz]')
axes[1,1].set_title('Spectral centroid shift vs Wiener λ')
axes[1,1].legend(fontsize=8)
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Results summary:**

| Kernel                        | RMSE filtered | RMSE reconstructed | SC_diff filtered [Hz] | SC_diff reconstructed [Hz] |
|:---|---:|---:|---:|---:|
| Gauss-5, σ=1                  | 0.00197       | 0.01692           | −188.7                | −2.3
| Gauss-11, σ=2                 | 0.00662       | 0.03364           | −573.4                | −51.3
| **Gauss-21, σ=3 (baseline)**  | **0.01126**   | **0.03312**      | **−961.1**             | **−201.6** |
| Gauss-51, σ=8                 | 0.01881       | 0.04326           | −1675.2               | −1415.5 |
| Gauss-101, σ=15                | 0.02279      | 0.05364          | −1842.7                | −1693 |

.

| λ                         | RMSE reconstructed    | SC_diff reconstructed [Hz]    | Δ RMSE vs baseline |
|:---|---:|---:|---:|
| 1×10⁻⁶                    | 0.03324               | −39.5                           | +0.00012 |
| 1×10⁻⁴                    | 0.03322               | −124.9                          | +0.00010 |
| **1×10⁻³ (baseline)**     | **0.03312**           | **−201.6**                      | **0.00** |
| 1×10⁻²                    | 0.03265               | −358.1                          | −0.00047 |
| 1×10⁻¹                    | 0.03060               | −736.7                          | −0.00252 |



## Day 15 – Analysis & Communication

*Focus: Analytical Interpretation and Domain-Specific Discussion*

<span style="background-color: #eeeeee;">*Guidelines: Based on the quantitative evaluation from the Evaluation day, critically analyze your findings in direct relation to your defined use case and selected signal or image. Structure your analysis into (1) observations, (2) interpretation, and (3) discussion. Observations must reference concrete quantitative results; interpretations must explain domain-specific implications; the discussion must critically assess the practical applicability, limitations, and risks of your implementation and evaluation approach. All arguments must explicitly connect to the original problem statement and use case. No new simulations or derivations may be introduced.*</span>

### Observations (3-5 sentences)
Focus: Describe measurable results only — no explanation.

> The quantitative evaluation shows that ____________ changes from ____________ to ____________ across configurations ____________.
The defined metric / extracted quantity indicates that ____________.
Performance differences are most pronounced when ____________.

### Interpretation (3-5 sentences)
Focus: Explain what the results mean for the application.

>In the context of ____________ [use case], these results imply that ____________.
The observed variation in ____________ affects ____________ because ____________.
This suggests that ____________ is particularly critical for achieving the objective defined in the problem statement.


### Discussion and Critical Reflection (4–6 sentences)
*Focus: Relate the quantitative findings to the requirements of the defined use case and assess practical adequacy.*

> For the defined use case, the configurations ____________ performed well because they achieved ____________, which aligns with the requirement of ____________.
In contrast, configurations ____________ showed reduced performance, leading to ____________ and limiting their suitability for this application.
The achieved performance level can be considered sufficient / insufficient for the use case because ____________.
The implementation and evaluation approach assumes ____________, which may be constrained under real-world conditions such as ____________.
To improve robustness or applicability, future work should address ____________ or refine ____________.

## Day 15 

### Final Reflections on MC
>Overall, the most successful aspect of MC1 was ____________, particularly in relation to ____________ within the defined use case. The implementation of ____________ worked well because ____________, leading to ____________.
If repeating this work, I would reconsider ____________ or refine ____________, as this could improve ____________ and reduce ____________. Certain challenges emerged during ____________, highlighting the importance of ____________ in practical signal/image analysis.
I was particularly impressed by ____________, especially how ____________ influenced ____________ in the context of the application.
Through this project, I learned that ____________, and I developed a clearer understanding of ____________, especially regarding the connection between theoretical principles and real-world constraints.

### Revisions
<span style="background-color: #fff3b0;"><strong>Template for Revisions:</strong> [your changes... ]</span>